In [2]:
!pip install spacy gradio pandas scikit-learn nltk deep-translator langdetect google-genai transformers accelerate -q
!python -m spacy download en_core_web_trf -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
from google.colab import drive
import pandas as pd
import re

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/twcs.csv')

print("total row:", len(df))
df.head()

Mounted at /content/drive
total row: 2811774


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [4]:
#total no of null row
print(df.isnull().sum())



tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64


In [5]:
import re


df = df[(df['inbound'] == True) & (df['in_response_to_tweet_id'].isna())].copy()
print("customer tweets:", len(df))


def handle(text):
    m = re.match(r"^\s*@(\w+)", str(text))
    if m:
        return m.group(1).lower()
    return ""

df['handle'] = df['text'].apply(handle)

print("top 25 handles:")
print(df['handle'].value_counts().head(25))

customer tweets: 787346
top 25 handles:
handle
                   313932
applesupport        30928
uber_support        14954
amazonhelp          14864
delta               13265
americanair         11994
115858              11744
spotifycares        10582
tesco               10427
british_airways     10328
askplaystation       9955
115850               9560
xboxsupport          9132
southwestair         9132
virgintrains         9130
115873               9036
comcastcares         8690
ask_spectrum         8013
gwrhelp              7336
safaricom_care       6968
115821               6923
115888               6554
hulu_support         6544
sainsburys           6001
115911               5953
Name: count, dtype: int64


In [6]:
handle_to_dept = {
    "applesupport": "Tech Support", "xboxsupport": "Tech Support",
    "askplaystation": "Tech Support", "spotifycares": "Tech Support",
    "hulu_support": "Tech Support", "microsofthelps": "Tech Support",
    "delta": "Travel", "americanair": "Travel", "british_airways": "Travel",
    "southwestair": "Travel", "uber_support": "Travel", "virginatlantic": "Travel",
    "amazonhelp": "Retail", "asktarget": "Retail", "tesco": "Retail",
    "sainsburys": "Retail", "asdaserviceteam": "Retail", "ihelpbestbuy": "Retail",
    "sprintcare": "Telecom", "verizonsupport": "Telecom", "tmobilehelp": "Telecom",
    "comcastcares": "Telecom", "ask_spectrum": "Telecom", "virginmedia": "Telecom",
    "chasesupport": "Billing", "ask_wellsfargo": "Billing", "bofa_help": "Billing",
    "askpaypal": "Billing", "askamex": "Billing", "askciti": "Billing",
}

df['department'] = df['handle'].map(handle_to_dept)
df = df.dropna(subset=['department'])

df = df.sample(n=min(150000, len(df)), random_state=42)
df = df[['text', 'inbound', 'department']]

print("distribution:")
print(df['department'].value_counts())
print("\ntotal rows:", len(df))
df.head()

distribution:
department
Tech Support    51717
Travel          45729
Retail          24496
Telecom         19605
Billing          8453
Name: count, dtype: int64

total rows: 150000


,text,inbound,department
1494881,@AskPlayStation Yo my psn got hacked and the d...,True,Tech Support
1923683,@AppleSupport The new apple update completely ...,True,Tech Support
923489,@Tesco received my dot com shopping today but ...,True,Retail
2248435,@AppleSupport my AirPods getting disconnected ...,True,Tech Support
998930,"@AskPayPal Hello, I would like to have any hel...",True,Billing


In [7]:

def clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\^\w+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean)
df = df[df['clean_text'].str.split().str.len() >= 5]

row = df[df['text'].str.contains('http', na=False)].iloc[0]
print("before:", row['text'])
print("after :", row['clean_text'])

before: @AppleSupport my AirPods getting disconnected after a minute! Let me know what to do https://t.co/OCGCaUwRYf
after : my airpods getting disconnected after a minute let me know what to do


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['department'], test_size=0.2, random_state=42,
    stratify=df['department']
)

# baseline naive bayes
nb = make_pipeline(
    TfidfVectorizer(stop_words='english', max_features=5000),
    MultinomialNB()
)
nb.fit(X_train, y_train)
print("nb accuracy:", round(accuracy_score(y_test, nb.predict(X_test)) * 100, 2), "%")

# logistic regression model
clf = make_pipeline(
    TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=25000,
        min_df=3,
        sublinear_tf=True
    ),
    LogisticRegression(max_iter=1000, C=2.0)
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("lr accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

nb accuracy: 82.84 %
lr accuracy: 86.63 %
              precision    recall  f1-score   support

     Billing       0.74      0.58      0.65      1588
      Retail       0.85      0.84      0.85      4770
Tech Support       0.87      0.92      0.89     10023
     Telecom       0.86      0.77      0.82      3717
      Travel       0.89      0.91      0.90      8915

    accuracy                           0.87     29013
   macro avg       0.84      0.80      0.82     29013
weighted avg       0.86      0.87      0.86     29013



In [9]:
import nltk
import spacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()     # between -1 to +1
ner = spacy.load("en_core_web_trf")    # for ententis
print("done")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


done


In [10]:
from textblob import TextBlob
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# manually labelled test complaints
sent_data = [
    ("Worst airline ever, my flight was cancelled and nobody helped!", "Very Negative"),
    ("My bank account was hacked and all my money is stolen, furious!", "Very Negative"),
    ("Terrible customer service, completely ruined my trip.", "Very Negative"),
    ("Horrible network speed, cannot even load a webpage.", "Very Negative"),
    ("My order from last Friday still has not arrived.", "Negative"),
    ("The app keeps crashing after the new update.", "Negative"),
    ("I was charged twice on my credit card statement.", "Negative"),
    ("Wifi disconnects every ten minutes at home.", "Negative"),
    ("Can you check the status of my flight tomorrow?", "Neutral"),
    ("I need to update my billing address on the account.", "Neutral"),
    ("What are your customer support hours on Sunday?", "Neutral"),
    ("Please share the tracking link for my package.", "Neutral"),
    ("Thank you for resolving my refund so quickly!", "Positive"),
    ("Great support from the technician today, issue is fixed.", "Positive"),
    ("Really happy with the fast delivery and smooth service.", "Positive"),
    ("Awesome help from your team, appreciate the quick reply!", "Positive"),
]

def vader_lbl(t):
    sc = sia.polarity_scores(t)['compound']
    if sc <= -0.5:
        return "Very Negative"
    elif sc < -0.05:
        return "Negative"
    elif sc > 0.05:
        return "Positive"
    return "Neutral"

def tb_lbl(t):
    pol = TextBlob(t).sentiment.polarity
    if pol <= -0.5:
        return "Very Negative"
    elif pol < -0.05:
        return "Negative"
    elif pol > 0.05:
        return "Positive"
    return "Neutral"

y_true = [y for _, y in sent_data]
y_vader = [vader_lbl(x) for x, _ in sent_data]
y_tb = [tb_lbl(x) for x, _ in sent_data]

lbls = ["Very Negative", "Negative", "Neutral", "Positive"]
print("vader accuracy:", round(accuracy_score(y_true, y_vader) * 100, 2), "%")
print("textblob accuracy:", round(accuracy_score(y_true, y_tb) * 100, 2), "%")
print("\nvader report:")
print(classification_report(y_true, y_vader, labels=lbls, zero_division=0))
print("confusion matrix (vader):")
print(confusion_matrix(y_true, y_vader, labels=lbls))

vader accuracy: 62.5 %
textblob accuracy: 62.5 %

vader report:
               precision    recall  f1-score   support

Very Negative       1.00      1.00      1.00         4
     Negative       0.00      0.00      0.00         4
      Neutral       0.40      0.50      0.44         4
     Positive       0.57      1.00      0.73         4

     accuracy                           0.62        16
    macro avg       0.49      0.62      0.54        16
 weighted avg       0.49      0.62      0.54        16

confusion matrix (vader):
[[4 0 0 0]
 [0 0 3 1]
 [0 0 2 2]
 [0 0 0 4]]


In [11]:
# manually annotated entity test set
ner_data = [
    ("My Amazon order never arrived in Pune last Friday.", {("Amazon", "ORG"), ("Pune", "GPE"), ("last Friday", "DATE")}),
    ("Delta Airlines cancelled my flight from Mumbai to London.", {("Delta Airlines", "ORG"), ("Mumbai", "GPE"), ("London", "GPE")}),
    ("Chase Bank charged me 50 dollars yesterday.", {("Chase Bank", "ORG"), ("50 dollars", "MONEY"), ("yesterday", "DATE")}),
    ("Apple Support said my iPhone warranty expired in January.", {("Apple Support", "ORG"), ("iPhone", "PRODUCT"), ("January", "DATE")}),
    ("Verizon internet has been down in New York since Monday.", {("Verizon", "ORG"), ("New York", "GPE"), ("Monday", "DATE")}),
    ("PayPal froze my account in Delhi on Tuesday.", {("PayPal", "ORG"), ("Delhi", "GPE"), ("Tuesday", "DATE")}),
    ("Uber driver overcharged me in Bangalore last night.", {("Uber", "ORG"), ("Bangalore", "GPE"), ("last night", "TIME")}),
    ("Spotify billed my Visa card twice this month.", {("Spotify", "ORG"), ("Visa", "ORG"), ("this month", "DATE")}),
]

tp = 0
fp = 0
fn = 0

for txt, gold in ner_data:
    doc = ner(txt)
    pred = {(e.text, e.label_) for e in doc.ents}
    tp += len(pred & gold)
    fp += len(pred - gold)
    fn += len(gold - pred)
    print("text:", txt)
    print("expected :", gold)
    print("predicted:", pred)
    print("-" * 30)

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0

print(f"ner precision: {round(prec * 100, 2)} %")
print(f"ner recall   : {round(rec * 100, 2)} %")
print(f"ner f1-score : {round(f1 * 100, 2)} %")

text: My Amazon order never arrived in Pune last Friday.
expected : {('Pune', 'GPE'), ('last Friday', 'DATE'), ('Amazon', 'ORG')}
predicted: {('last Friday', 'DATE'), ('Pune', 'GPE'), ('Amazon', 'ORG')}
------------------------------
text: Delta Airlines cancelled my flight from Mumbai to London.
expected : {('Mumbai', 'GPE'), ('London', 'GPE'), ('Delta Airlines', 'ORG')}
predicted: {('Mumbai', 'GPE'), ('London', 'GPE'), ('Delta Airlines', 'ORG')}
------------------------------
text: Chase Bank charged me 50 dollars yesterday.
expected : {('50 dollars', 'MONEY'), ('Chase Bank', 'ORG'), ('yesterday', 'DATE')}
predicted: {('50 dollars', 'MONEY'), ('Chase Bank', 'ORG'), ('yesterday', 'DATE')}
------------------------------
text: Apple Support said my iPhone warranty expired in January.
expected : {('Apple Support', 'ORG'), ('iPhone', 'PRODUCT'), ('January', 'DATE')}
predicted: {('iPhone', 'PRODUCT'), ('Apple Support', 'ORG'), ('January', 'DATE')}
------------------------------
text: Veriz

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

sample_tweets = df['clean_text'].sample(min(50000, len(df)), random_state=42)

lda_vec = CountVectorizer(max_df=0.9, min_df=10, stop_words='english', max_features=2000)
dtm = lda_vec.fit_transform(sample_tweets)

lda = LatentDirichletAllocation(n_components=5, learning_method='online', random_state=42)
lda.fit(dtm)

words = lda_vec.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-8:][::-1]
    top_words = [words[idx] for idx in top_indices]
    print(f"Topic {i + 1}:", ", ".join(top_words))

Topic 1: internet, going, order, just, want, guys, days, like
Topic 2: account, help, dont, email, card, hi, does, working
Topic 3: flight, service, trying, im, customer, need, help, time
Topic 4: phone, app, iphone, help, update, fix, ios, new
Topic 5: having, store, im, issues, battery, received, 10, minutes


In [13]:
import torch
from langdetect import detect
from deep_translator import GoogleTranslator
from google import genai
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM

#gemini api
try:
    key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=key)
    print("gemini ready")
except Exception as e:
    print("gemini error:", e)
    client = None

g_model = "gemini-3.5-flash-lite"

#qwen model
q_name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(q_name)
qwen = AutoModelForCausalLM.from_pretrained(q_name, device_map="auto")
print("qwen loaded on:", qwen.device)

gemini ready


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

qwen loaded on: cuda:0


In [20]:
#this cell is about if the input is in other lang it will translate into english

def trans(txt):
    try:
        lang = detect(txt)
    except Exception:
        lang = "en"

    if lang != "en":
        try:
            eng = GoogleTranslator(source='auto', target='en').translate(txt)
            if eng and eng.strip():
                return lang, eng.strip(), True
        except Exception as e:
            if client is not None:
                try:
                    res = client.models.generate_content(
                        model=g_model,
                        contents=f"translate this complaint to english only:\n{txt}"
                    )
                    if res.text and res.text.strip():
                        return lang, res.text.strip(), True
                except Exception as e2:
                    print("trans error:", e2)
            else:
                print("trans error:", e)
        return lang, "", False

    return lang, txt, True


# internal note for support team (qwen)
def brief(txt, dept, prio, ents):
    p = f"""write a short plain-text note for the {dept} team. no markdown, stars, or hashtags.
complaint: {txt}
priority: {prio}
entities: {ents}

use this exact format:
summary: <1 short sentence about the customer issue>
next steps:
- <first check>
- <second check>"""

    try:
        msg = [
            {"role": "system", "content": "you write short plain-text support notes without markdown, stars, or hashtags."},
            {"role": "user", "content": p}
        ]
        chat = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        inp = tok([chat], return_tensors="pt").to(qwen.device)

        with torch.no_grad():
            out_ids = qwen.generate(
                **inp,
                max_new_tokens=95,
                temperature=0.2,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tok.eos_token_id
            )

        res = tok.decode(out_ids[0][inp.input_ids.shape[-1]:], skip_special_tokens=True)
        res = re.sub(r'\*+', '', res)
        res = re.sub(r'#+\s*', '', res)
        return res.strip()
    except Exception as e:
        print("qwen error:", e)
        return f"summary: {dept} issue ({prio})"


# polite reply for customer (gemini)
def reply(txt, dept, sent, prio, ents):
    p = f"""write a polite 3 sentence customer support reply for the {dept} team.
mention key details, match urgency if priority is high, and do not make up order numbers or refunds.
complaint: {txt}
sentiment: {sent}
priority: {prio}
entities: {ents}"""

    if client is not None:
        try:
            res = client.models.generate_content(model=g_model, contents=p)
            return res.text.strip()
        except Exception as e:
            print("gemini error:", e)

    return f"sorry for the trouble. your issue is sent to {dept} with {prio} and our team will contact you soon."

In [21]:
# main pipeline
def proc(txt):
    if txt.strip() == "":
        return None, "", "", "", "", "", "", ""

    lang, eng, ok = trans(txt)
    if not ok or eng.strip() == "":
        return None, "error", "error", "none", lang, "translation failed", "error: translation failed", "unable to translate complaint. please try again or enter in english."

    eng_show = "not needed" if lang == "en" else eng
    cln = clean(eng)
    if not re.search(r'[a-z]', cln):
        return None, "error", "error", "none", lang, eng_show, "error: no valid words found", "please enter a valid complaint with words."

    # department probabilities
    probs = clf.predict_proba([cln])[0]
    scores = {c: float(p) for c, p in zip(clf.classes_, probs)}
    dept = clf.predict([cln])[0]

    # sentiment and priority
    sc = sia.polarity_scores(eng)['compound']
    if sc <= -0.5:
        sent = "Very Negative"
        prio = "High Priority"
    elif sc < -0.05:
        sent = "Negative"
        prio = "Medium Priority"
    elif sc > 0.05:
        sent = "Positive"
        prio = "Normal"
    else:
        sent = "Neutral"
        prio = "Normal"

    # named entities
    doc = ner(eng)
    ents = [f"{e.label_}: {e.text}" for e in doc.ents]
    ent_str = "none" if len(ents) == 0 else "\n".join(ents)

    b_txt = brief(eng, dept, prio, ent_str)
    r_txt = reply(eng, dept, sent, prio, ent_str)

    return scores, f"{sent} ({sc})", prio, ent_str, lang, eng_show, b_txt, r_txt

print("done")

done


In [17]:

print(proc("My Amazon order from last Friday never arrived in Pune and nobody is helping me, worst service!"))
print("=" * 40)
print(proc("माझी डेल्टा एअरलाइन्सची फ्लाइट रद्द झाली आणि माझी बॅग हरवली आहे."))
print("=" * 40)
print(proc("मेरा इंटरनेट कल रात से काम नहीं कर रहा है और कोई मदद नहीं कर रहा।"))

({'Billing': 0.0028096910604267504, 'Retail': 0.9780849691021654, 'Tech Support': 0.001062948486870994, 'Telecom': 0.00774576430521361, 'Travel': 0.010296627045323345}, 'Negative (-0.3595)', 'Medium Priority', 'ORG: Amazon\nDATE: last Friday\nGPE: Pune', 'en', 'not needed', "Summary: The customer's Amazon order from last Friday has not arrived in Pune. This is causing inconvenience and frustration.\n\nNext Steps:\n- Check with Amazon's customer support to understand the status of the order.\n- Reach out to the local store in Pune if possible to inquire about delivery details.", "We sincerely apologize for the delay with your Amazon order from last Friday to Pune and understand your frustration with the support you've received so far. To help us resolve this immediately, could you please reply with your specific order number and any tracking details you have? Our retail team is prioritizing your case and will investigate the delivery status with the courier right away.")
({'Billing': 8.

In [18]:
import joblib
from google.colab import files

fname = 'nlp_model.pkl'
joblib.dump(clf, fname)

print(f"done {fname}")
files.download(fname)

done nlp_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
import gradio as gr

with gr.Blocks(title="complaint routing system") as ui:

    gr.Markdown("# Complaint Routing System")
    gr.Markdown("Enter a complaint in english, hindi or marathi")

    with gr.Row():
        with gr.Column():
            inp = gr.Textbox(lines=6, label="complaint")
            with gr.Row():
                clear_btn = gr.Button("Clear")
                run_btn = gr.Button("Submit", variant="primary")

        with gr.Column():
            dept_out = gr.Label(label="department", num_top_classes=5)
            with gr.Row():
                sent_out = gr.Textbox(label="sentiment")
                prio_out = gr.Textbox(label="priority")
            ents_out = gr.Textbox(label="entities", lines=3)

    with gr.Row():
        lang_out = gr.Textbox(label="language")
        eng_out = gr.Textbox(label="english translation")

    with gr.Row():
        brief_out = gr.Textbox(label="internal brief", lines=6)
        reply_out = gr.Textbox(label="draft reply", lines=6)

    outs = [dept_out, sent_out, prio_out, ents_out,
            lang_out, eng_out, brief_out, reply_out]

    run_btn.click(proc, inputs=inp, outputs=outs)
    clear_btn.click(lambda: [""] + [None] * 8, None, [inp] + outs)

ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55854106f70b846c89.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
